# Phase A: Question Decomposition

**Journey:** grounded-research v2 — close the competitive gap with Perplexity deep research  
**Purpose:** Add structured question decomposition before search and analysis  
**Mode:** Planning (stubs and fixtures — no live LLM calls yet)

**Why this matters:** Perplexity's hidden planning system decomposes questions before searching. We send the monolithic question to search and analysts as-is. This is the #1 competitive gap (Perplexity beats us on completeness 5 vs 4).

**Related docs:**
- `docs/ROADMAP_V2.md` — phase sequencing and gates
- `docs/FEATURE_STATUS.md` — scorecard items #1, #2, #4, #5
- `docs/COMPETITIVE_ANALYSIS.md` — what Perplexity does better
- `src/grounded_research/models.py` — domain model
- `prompts/analyst.yaml` — current analyst prompt (receives raw question)

## Phase A.1: Question Reformulation & Decomposition

**Input:** Raw user question string  
**Output:** `QuestionDecomposition` (precise core question, 2-6 typed sub-questions, optimization axes, research plan)  
**Status:** `planned`  
**Execution mode:** `stub`

**Acceptance criteria:**
- Core question is a precise, searchable reformulation of the raw input
- Sub-questions are typed (factual, causal, comparative, evaluative, scope)
- Sub-questions collectively cover the full question (no major facet missing)
- Each sub-question has a falsification target (what evidence would disprove it)
- Optimization axes name the key tradeoffs a reader needs to evaluate

In [ ]:
# Schema: QuestionDecomposition
# This is the contract — implementation will use these exact types.

from __future__ import annotations
from pydantic import BaseModel, Field
from typing import Literal

SubQuestionType = Literal["factual", "causal", "comparative", "evaluative", "scope"]

class SubQuestion(BaseModel):
    """One dimension of the research question."""
    text: str = Field(description="The sub-question, phrased as a searchable question.")
    type: SubQuestionType = Field(description="What kind of question this is. Factual: what happened. Causal: why/how. Comparative: X vs Y. Evaluative: how good/effective. Scope: what's the boundary.")
    falsification_target: str = Field(description="What evidence would disprove or weaken the expected answer to this sub-question.")

class QuestionDecomposition(BaseModel):
    """Structured decomposition of a research question into searchable sub-questions."""
    core_question: str = Field(description="Precise reformulation of the user's raw question. Should be unambiguous and searchable.")
    sub_questions: list[SubQuestion] = Field(description="2-6 typed sub-questions that collectively cover the full question.", min_length=2, max_length=6)
    optimization_axes: list[str] = Field(description="The 2-4 key tradeoffs or dimensions a reader needs to evaluate. E.g., 'short-term revenue impact vs long-term structural change'.", min_length=1, max_length=4)
    research_plan: str = Field(description="Brief plan: what to search for, which source types matter most, what critical evidence to prioritize.")

print("Schema defined. Fields:")
for name, field in QuestionDecomposition.model_fields.items():
    print(f"  {name}: {field.annotation}")
print(f"\nSubQuestionType options: {SubQuestionType.__args__}")

In [ ]:
# Stub: What a decomposition looks like for our test question
# This is the fixture — what we expect the LLM to produce.

SANCTIONS_QUESTION = "What sanctions has the EU imposed on Russia since February 2022, and how effective have they been at reducing Russian oil and gas revenue?"

stub_decomposition = QuestionDecomposition(
    core_question="What specific energy sanctions has the EU imposed on Russia since February 2022, and to what extent have these sanctions reduced Russia's oil and gas export revenue?",
    sub_questions=[
        SubQuestion(
            text="What specific sanctions has the EU imposed on Russian oil exports since February 2022, including import bans, price caps, and enforcement mechanisms?",
            type="factual",
            falsification_target="Evidence that major reported sanctions (e.g., oil import ban, price cap) were not actually implemented or were reversed."
        ),
        SubQuestion(
            text="What specific sanctions has the EU imposed on Russian gas and LNG exports, and how do they differ from oil sanctions in timing and scope?",
            type="factual",
            falsification_target="Evidence that gas sanctions are as comprehensive as oil sanctions (they are widely reported as weaker/later)."
        ),
        SubQuestion(
            text="How much have Russia's oil and gas export revenues declined since sanctions were imposed, and what portion of that decline is attributable to sanctions vs other factors (market prices, self-sanctioning, demand shifts)?",
            type="causal",
            falsification_target="Evidence that revenue declines are primarily explained by global oil price drops or demand destruction rather than sanctions."
        ),
        SubQuestion(
            text="How effectively has Russia circumvented EU energy sanctions through shadow fleets, third-country intermediaries, and price cap evasion?",
            type="evaluative",
            falsification_target="Evidence that circumvention is minimal and the price cap is binding on most Russian oil exports."
        ),
        SubQuestion(
            text="How do EU energy sanctions compare in effectiveness to the broader G7 sanctions regime, and what is the EU's relative contribution to revenue reduction?",
            type="comparative",
            falsification_target="Evidence that non-EU sanctions (US, UK) account for most of the revenue impact, with EU sanctions being marginal."
        ),
    ],
    optimization_axes=[
        "Immediate revenue reduction vs long-term structural energy dependence shift",
        "Sanctions comprehensiveness vs political feasibility within EU member states",
        "Direct revenue impact vs circumvention-adjusted net effectiveness",
    ],
    research_plan=(
        "Prioritize: (1) EU Council sanctions package announcements for factual sanctions inventory, "
        "(2) CREA and IEA data for revenue figures, (3) academic/think-tank analysis for causal attribution, "
        "(4) investigative journalism on shadow fleet and circumvention. Critical evidence: month-by-month "
        "Russian energy revenue data with sanctions timeline overlay."
    ),
)

print(f"Core question: {stub_decomposition.core_question[:80]}...")
print(f"Sub-questions: {len(stub_decomposition.sub_questions)}")
for sq in stub_decomposition.sub_questions:
    print(f"  [{sq.type}] {sq.text[:70]}...")
print(f"Axes: {stub_decomposition.optimization_axes}")
print(f"Plan: {stub_decomposition.research_plan[:80]}...")

## Phase A.2: Sub-Question-Driven Search

**Input:** `QuestionDecomposition` + search config  
**Output:** `EvidenceBundle` with evidence tagged by sub-question  
**Status:** `planned`  
**Execution mode:** `stub`

**Acceptance criteria:**
- Each sub-question generates its own search queries (not just one flat list)
- Evidence items are tagged with which sub-question they address
- Evidence coverage is checked per sub-question (≥ 2 sources each)
- Total source count remains configurable (default 50)

**Key design decision:** Search queries are generated per sub-question, but the source pool is shared and deduplicated. A source found via sub-question #1 may also contain evidence for sub-question #3. The tagging happens at the evidence item level, not the source level.

**What changes in `collect.py`:**
- `generate_search_queries()` takes a list of sub-questions instead of a raw question
- Generates 3-5 queries per sub-question (was 15 for the whole question)
- Evidence items get a new field: `sub_question_ids: list[str]`
- Post-collection: check coverage per sub-question, flag gaps

In [ ]:
# Stub: Sub-question-driven search query generation
# Shows the contract — each sub-question produces its own query set

def generate_queries_per_subquestion(decomposition: QuestionDecomposition) -> dict[str, list[str]]:
    """Stub: Generate search queries grouped by sub-question.
    
    In live mode, each sub-question gets an LLM call to produce 3-5 queries.
    Here we show the expected shape.
    """
    queries_by_sq: dict[str, list[str]] = {}
    
    for sq in decomposition.sub_questions:
        # In live: LLM generates 3-5 queries per sub-question
        # Stub: show the pattern
        queries_by_sq[sq.text[:50]] = [
            f"{sq.text}",
            f"{sq.text} evidence data",
            f"{sq.text} criticism counterargument",
        ]
    
    return queries_by_sq

stub_queries = generate_queries_per_subquestion(stub_decomposition)
print(f"Sub-questions: {len(stub_queries)}")
for sq_key, queries in stub_queries.items():
    print(f"\n  [{sq_key}...]")
    for q in queries:
        print(f"    → {q[:70]}...")
print(f"\nTotal queries: {sum(len(qs) for qs in stub_queries.values())}")

## Phase A.3: Decomposition-Aware Analysis

**Input:** `QuestionDecomposition` + `EvidenceBundle`  
**Output:** `AnalystRun` (unchanged schema, but analysts receive sub-questions as structured context)  
**Status:** `planned`  
**Execution mode:** `planned`

**Acceptance criteria:**
- Analyst prompt receives sub-questions as structured context (not just raw question)
- Analysts produce claims that map to specific sub-questions
- Analysts address optimization axes in their recommendations
- Analysis is more structured and complete than monolithic question approach

**What changes in `prompts/analyst.yaml`:**
- User message includes sub-questions with types and falsification targets
- Instruction: "Address each sub-question. For each, cite evidence and note where evidence is weak."
- Instruction: "Consider the optimization axes when forming your recommendation."

**What does NOT change:**
- AnalystRun schema (no new fields needed — claims already have evidence_ids)
- Number of analysts (3)
- Model assignment and frame assignment (unchanged)
- Claim extraction, dedup, dispute detection (all downstream, unchanged)

## Phase A.4: Decomposition-Aware Synthesis

**Input:** `QuestionDecomposition` + `ClaimLedger` + `EvidenceBundle`  
**Output:** Long-form report (unchanged format, but uses sub-questions and axes for structure)  
**Status:** `planned`  
**Execution mode:** `planned`

**Acceptance criteria:**
- Synthesis prompt receives optimization axes as explicit structure
- Report "key distinctions" align with optimization axes (not invented from scratch)
- Report addresses each sub-question (coverage check)
- Evidence gaps section references specific sub-questions with weak coverage

**What changes in `prompts/long_report.yaml`:**
- User message includes optimization axes and sub-questions
- Instruction: "Use the optimization axes as the organizing framework for your key distinctions."
- Instruction: "Ensure every sub-question is addressed. Flag sub-questions where evidence was insufficient."

**What does NOT change:**
- FinalReport schema
- Report sections (summary, distinctions, evidence, gaps, verdict, alternatives, conditions)
- Grounding validation

## Gate: Phase A Complete

**Measured by:** Fair comparison (compare_fair.py) of Phase A pipeline vs v1 pipeline vs Perplexity deep, all on EU sanctions question.

**Pass if:**
- Completeness score ≥ 5 (was 4 — the dimension Perplexity beat us on)
- Analyst claims cover all sub-questions (no sub-question with 0 claims)
- Report "key distinctions" visibly align with optimization axes
- Overall fair score ≥ 23/25 (was 21/25)

**Fail if:**
- Decomposition adds cost but doesn't improve completeness
- Sub-questions are redundant or poorly typed
- Pipeline breaks backward compatibility (old questions without decomposition must still work)

## Implementation Sequence

When this notebook is promoted to `live`:

1. Add `QuestionDecomposition`, `SubQuestion` to `models.py`
2. Add `prompts/decompose.yaml` template
3. Add `decompose_question()` function in new `src/grounded_research/decompose.py`
4. Modify `collect.py`: `generate_search_queries()` accepts optional sub-questions
5. Modify `prompts/analyst.yaml`: include sub-questions in user message
6. Modify `prompts/long_report.yaml`: include optimization axes
7. Modify `engine.py`: wire decomposition step before collection
8. Backward compatible: if no decomposition, behaves as v1
9. Run comparison, measure gate criteria

## Open Decisions (resolve before implementing)

1. **Decomposition model**: Same as analyst model (gemini-2.5-flash) or synthesis model? Recommend analyst model — decomposition is reasoning, not generation.
2. **Config key**: `models.decomposition` in config.yaml
3. **Backward compatibility**: Add `--no-decompose` flag to engine.py, or make decomposition always-on? Recommend always-on with the flag for A/B testing.
4. **Sub-question ID format**: `SQ-{hash}` prefix (consistent with C-, E-, S- pattern)? Or just list index?